# regnet_y_800mf: seeds × datasets (generalization check)

Follow-up to `experiments.ipynb`. The single-seed grid there picks the epoch with the lowest val EER out of 30,
and at constant lr 1e-3 val EER jumps by up to 30 pp between epochs, so a single run's best epoch is partly luck.
Here a few candidate configs are rerun with **3 seeds** on **2 datasets** (the bigger v4 and our_voices) to see which config is stable and whether
the best config depends on the dataset.

Runs land in `runs/dnlvtk/regnet_gen_<dataset>/<run_name>/` (one topic per dataset, so identical configs on different
data never share a folder). `export_onnx=false` (bookkeeping only, not part of the run hash).

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd

from iml.cli import check_run_data, data_fingerprint, load_splits, record_run_data, summary_table
from iml.config import ExperimentConfig
from iml.imbalance import imbalance_report
from iml.runs import load_run, run_dir
from iml.tracking import Tracker
from iml.train import fit

MEMBER = "dnlvtk"
MODEL = "regnet_y_800mf"
DATASETS = {
    "v4": ROOT / "data/training_dataset_v4",  # 1148 train (949 neg / 199 pos)
    "our_voices": ROOT / "data/our_voices",  # 647 train (56 / 591); only 7 val / 9 test negatives
}
SEEDS = [42, 7, 123]
pd.set_option("display.width", 250)

## 1. Configs

In [ ]:
BASE = ExperimentConfig(model=MODEL, member=MEMBER, pretrained=True, optimizer="adamw", epochs=30,
                        dropout=0.3, class_weight="balanced", export_onnx=False)
EXPLICIT = {
    "A_const_lr1e-3": ["lr=1e-3"],  # best single run on v4 (7a3418, lr 1.1e-3)
    "B_cos_wd_lr1e-3": ["lr=1e-3", "scheduler=cosine", "weight_decay=0.05"],
    "C_cos_wd_lr3e-4": ["lr=3e-4", "scheduler=cosine", "weight_decay=0.05"],
    "D_cos_wd_lr1e-3_noCW": ["lr=1e-3", "scheduler=cosine", "weight_decay=0.05", "class_weight=none"],
}

## 2. Train (cached)

In [ ]:
records, meta = [], []
for ds, path in DATASETS.items():
    fp = data_fingerprint(path)
    loaders = None
    for name, ov in EXPLICIT.items():
        for seed in SEEDS:
            cfg = BASE.with_overrides(*ov, f"seed={seed}", f"topic=regnet_gen_{ds}")
            out = run_dir(cfg)
            check_run_data(out, fp, overwrite=False)
            if (out / "metrics.json").exists():
                rec = load_run(out)
            else:
                loaders = loaders or load_splits(cfg, data_dir=path)
                rec = fit(cfg, loaders, tracker=Tracker(enabled=False))
                record_run_data(out, fp)
            records.append(rec)
            meta.append({"dataset": ds, "config": name, "seed": seed, "path": out})
    if loaders is not None:
        print(ds, "train imbalance ratio:", imbalance_report(loaders["train"])["ratio"])

## 3. Compare: mean ± std over seeds, plus late-epoch stability

In [ ]:
import json

rows = []
for m in meta:
    mj = json.loads((m["path"] / "metrics.json").read_text())
    h = pd.read_csv(m["path"] / "history.csv")
    rows.append({**{k: m[k] for k in ("dataset", "config", "seed")}, "best_epoch": mj["best_epoch"],
                 **{f"{s}_{k}": mj[s][k] for s in ("train", "val", "test") for k in ("eer", "far", "frr", "macro_f1")},
                 "val_eer_last10": h.val_eer.tail(10).mean(), "val_loss_last10": h.val_loss.tail(10).mean(),
                 "train_loss_last": h.train_loss.iloc[-1]})
per_run = pd.DataFrame(rows)
cols = ["val_eer", "test_eer", "test_far", "test_frr", "test_macro_f1", "train_eer", "val_eer_last10"]
agg = per_run.groupby(["dataset", "config"])[cols].agg(["mean", "std"]).round(3)
agg

In [ ]:
out = ROOT / "runs" / MEMBER / "regnet_gen_comparison"
out.mkdir(parents=True, exist_ok=True)
per_run.to_csv(out / "per_run.csv", index=False)
agg.to_csv(out / "agg.csv")

### Conclusion

_See `reports/regnet/README.md`._